Question 1

id: load_nyc_taxi_csv_to_bigquery
namespace: data.eng

inputs:
  - id: project_id
    type: STRING
    defaults: your-gcp-project
  - id: gcs_csv_uri
    type: STRING
    defaults: gs://your-bucket/nyc_taxi/yellow_tripdata_2024-01.csv
  - id: bq_table
    type: STRING
    defaults: your-gcp-project.nyc_taxi.yellow_trips
  - id: location
    type: STRING
    defaults: US

tasks:
  - id: load_csv_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.LoadFromGcs
    projectId: "{{ inputs.project_id }}"
    location: "{{ inputs.location }}"
    from:
      - "{{ inputs.gcs_csv_uri }}"
    destinationTable: "{{ inputs.bq_table }}"
    format: CSV
    createDisposition: CREATE_IF_NEEDED
    writeDisposition: WRITE_TRUNCATE
    autodetect: false
    ignoreUnknownValues: true
    maxBadRecords: 10
    csvOptions:
      skipLeadingRows: 1
      fieldDelimiter: ","
      encoding: UTF-8
      allowQuotedNewLines: true
    schema:
      fields:
        - name: VendorID
          type: INT64
        - name: tpep_pickup_datetime
          type: TIMESTAMP
        - name: tpep_dropoff_datetime
          type: TIMESTAMP
        - name: passenger_count
          type: FLOAT64
        - name: trip_distance
          type: FLOAT64
        - name: RatecodeID
          type: FLOAT64
        - name: store_and_fwd_flag
          type: STRING
        - name: PULocationID
          type: INT64
        - name: DOLocationID
          type: INT64
        - name: payment_type
          type: INT64
        - name: fare_amount
          type: FLOAT64
        - name: extra
          type: FLOAT64
        - name: mta_tax
          type: FLOAT64
        - name: tip_amount
          type: FLOAT64
        - name: tolls_amount
          type: FLOAT64
        - name: improvement_surcharge
          type: FLOAT64
        - name: total_amount
          type: FLOAT64
        - name: congestion_surcharge
          type: FLOAT64
        - name: Airport_fee
          type: FLOAT64

id: nyc-taxi-data-load
namespace: dev

tasks:
  - id: download-taxi-data
    type: io.kestra.plugin.core.http.Download
    uri: https://storage.googleapis.com/cloud-samples-data/bigquery/sample-data/nyc_taxi_trips.csv
    saveAs: nyc_taxi_trips.csv

  - id: load-to-bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs['download-taxi-data'].uri }}"
    destinationTable: "your_project_id.your_dataset_id.nyc_taxi_trips" # Update with your BigQuery project, dataset, and table
    format: CSV
    autodetect: true
    csvOptions:
      skipLeadingRows: 1
      fieldDelimiter: ","
    projectId: "your_gcp_project_id" # Update with your GCP Project ID
    location: "US" # Update with your BigQuery dataset location if different
    createDisposition: CREATE_IF_NEEDED
    writeDisposition: WRITE_TRUNCATE

Question 2

id: 1_chat_without_rag
namespace: zoomcamp

description: |
  This flow demonstrates what happens when you query an LLM WITHOUT RAG.
  The model can only rely on its training data, which may be outdated or incomplete.

  After running this, check out 2_chat_with_rag.yaml to see how RAG fixes these issues!

tasks:
  - id: chat_without_rag
    type: io.kestra.plugin.ai.completion.ChatCompletion
    description: Query about Kestra 1.1 features WITHOUT RAG
    provider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-2.5-flash
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    messages:
      - type: USER
        content: |
          Which features were released in Kestra 1.1?
          Please list at least 5 major features with brief descriptions.

  - id: log_results
    type: io.kestra.plugin.core.log.Log
    message: |
      ❌ Response WITHOUT RAG (no retrieved context):
      {{ outputs.chat_without_rag.textOutput }}

      🤔 Did you notice that this response seems to be:
      - Incorrect?
      - Vague/generic?
      - Listing features that haven't been added in exactly this version but rather a long time ago?

      👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.

2026-07-05T12:01:01.317040Z INFO ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting new features, focusing on improving the developer experience, adding more robust functionalities, and enhancing observability. Here are 5 major features with brief descriptions:

1.  **Declarative Python API:**
    *   **Description:** This was a significant addition, allowing users to define Kestra flows directly within Python code using a declarative style. Instead of writing YAML files, developers can now leverage the power of Python to construct their workflows, making them more programmatic, testable, and maintainable. It opens up opportunities for more complex logic and easier integration with existing Python projects.

2.  **Plugin for VS Code:**
    *   **Description:** Kestra 1.1 brought an official Visual Studio Code extension. This plugin provides IDE-level support for Kestra users, including features like syntax highlighting, auto-completion, schema validation, and potentially even direct deployment capabilities for Kestra flows. This greatly enhances the developer experience by making it easier to write, debug, and manage Kestra workflows within a familiar and powerful editor.

3.  **Outputs on Tasks and Task Runs:**
    *   **Description:** This feature enhanced the observability and data passing capabilities within Kestra flows. It allows tasks to explicitly define and store outputs, which can then be accessed by subsequent tasks in the flow. Previously, data passing might have relied more on internal storage or less structured methods. With structured outputs, it's easier to understand the data generated by each task and use it downstream, improving data lineage and flow logic.

4.  **Flow Template Discovery & Management:**
    *   **Description:** Kestra 1.1 introduced better mechanisms for discovering and managing flow templates. This feature helps users reuse common workflow patterns and components more effectively. It makes it easier to find, import, and apply pre-defined flow structures, promoting consistency and reducing boilerplate code across different projects and teams. This is crucial for scaling Kestra usage within organizations.

5.  **Blueprint Storage from a Git Repository:**
    *   **Description:** This feature allows Kestra to store and manage "blueprints" (reusable flow components or templates) directly from a Git repository. This means that teams can version control their blueprints, collaborate on them using standard Git workflows, and ensure that their reusable components are always up-to-date and accessible. It integrates Kestra more deeply with existing CI/CD practices and promotes best practices for code management.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.

id: 2_chat_with_rag
namespace: zoomcamp

description: |
  This flow demonstrates RAG (Retrieval Augmented Generation) by ingesting Kestra release documentation and using it to answer questions accurately.

  Compare this with 1_chat_without_rag.yaml to see the difference RAG makes!

tasks:
  - id: ingest_release_notes
    type: io.kestra.plugin.ai.rag.IngestDocument
    description: Ingest Kestra 1.1 release notes to create embeddings
    provider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-embedding-001
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddings:
      type: io.kestra.plugin.ai.embeddings.KestraKVStore
    drop: true
    fromExternalURLs:
      - https://raw.githubusercontent.com/kestra-io/docs/refs/heads/main/src/contents/blogs/release-1-1/index.md

  - id: chat_with_rag
    type: io.kestra.plugin.ai.rag.ChatCompletion
    description: Query about Kestra 1.1 features with RAG context
    chatProvider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-2.5-flash
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddingProvider:
      type: io.kestra.plugin.ai.provider.GoogleGemini
      modelName: gemini-embedding-001
      apiKey: "{{ secret('GEMINI_API_KEY') }}"
    embeddings:
      type: io.kestra.plugin.ai.embeddings.KestraKVStore
    systemMessage: |
      You are a helpful assistant that answers questions about Kestra.
      Use the provided documentation to give accurate, specific answers.
      If you don't find the information in the context, say so.
    prompt: |
      Which features were released in Kestra 1.1?
      Please list at least 5 major features with brief descriptions.

  - id: log_results
    type: io.kestra.plugin.core.log.Log
    message: |
      ✅ RAG Response (with retrieved context):
      {{ outputs.chat_with_rag.textOutput }}

      🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!

2026-07-05T12:01:34.444568Z INFO ✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features, including:

1.  **New Filters**: The UI filters across Kestra were completely redesigned based on user feedback to be more intuitive and powerful. Users can now choose from explicit filter options, reset filters with a single click, save frequently used combinations, and customize table columns.
2.  **No-Code Dashboard Editor**: This feature extends the no-code multi-panel editor to custom dashboards, allowing users to build and customize dashboards directly from the UI without writing YAML. It supports creating data sources, visualizations, and charts using form-based tabs.
3.  **Multi-Agent AI Systems**: AI agents in Kestra can now use other AI agents as tools, enabling sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
4.  **Fix with AI**: When task runs fail, Kestra 1.1 provides AI-powered suggestions to help users quickly diagnose and resolve issues, offering intelligent recommendations for fixing problems.
5.  **Human Task**: For Enterprise Edition users, this new feature allows adding manual approval steps to workflows. When an execution reaches a human task, it pauses until designated users or group members approve and resume it, enabling human-in-the-loop workflows.

Other notable features include improved air-gapped support, a new `MailReceivedTrigger`, enhanced file detection triggers, custom app branding (Enterprise Edition), and dozens of new plugins across various categories like data, SaaS, cloud, and AI.

🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!

Question 3

id: 4_simple_agent
namespace: zoomcamp
description: |
  This flow demonstrates a basic AI agent that summarizes text with controllable length and language. It shows:
  - How to structure agent prompts
  - How to chain multiple agent tasks
  - How to use pluginDefaults to avoid repetition
  - How to track token usage for cost monitoring
inputs:
  - id: summary_length
    displayName: Summary Length
    type: SELECT
    defaults: medium
    values:
      - short
      - medium
      - long
  - id: language
    displayName: Language
    type: SELECT
    defaults: en
    values:
      - en
      - fr
      - de
      - es
      - it
      - pt
      - ja
  - id: text
    type: STRING
    displayName: Text to summarize
    defaults: |
      Kestra is an open-source orchestration platform that allows you to define workflows declaratively in YAML. It enables both developers and non-developers to automate tasks through a no-code interface, while keeping everything versioned, governed, secure, and auditable. Kestra extends easily for custom use cases through plugins and custom scripts.

      Kestra follows a "start simple and grow as needed" philosophy. You can schedule a basic workflow in a few minutes, then later add Python scripts, Docker containers, or complex branching logic if the situation requires it. This makes Kestra ideal for data engineering, ETL pipelines, business process automation, and more.

      In LLM Zoomcamp, we learn how to build production-ready LLM applications using RAG, vector search, agents, and evaluation. In this bonus module, we're exploring how AI can accelerate workflow development through AI Copilot, RAG, and autonomous agents.
tasks:
  - id: multilingual_agent
    type: io.kestra.plugin.ai.agent.AIAgent
    description: Generate summary in requested language and length
    systemMessage: |
      You are a precise technical assistant.
      Produce a {{ inputs.summary_length }} summary in {{ inputs.language }}.
      Keep it factual, remove fluff, and avoid marketing language.
      If the input is empty or non-text, return a one-sentence explanation.

      Output format guidelines:
      - For 'short': 1-2 sentences
      - For 'medium': 2-5 sentences
      - For 'long': 1-3 paragraphs
    prompt: |
      Summarize the following content: {{ inputs.text }}
  - id: english_brevity
    type: io.kestra.plugin.ai.agent.AIAgent
    prompt: |
      Generate exactly 3 sentence English summary of the following:
      "{{ outputs.multilingual_agent.textOutput }}"
  - id: log_token_usage
    type: io.kestra.plugin.core.log.Log
    message: |
      📊 Token Usage Summary:

      Multilingual Agent:
      - Input tokens: {{ outputs.multilingual_agent.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.multilingual_agent.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.multilingual_agent.tokenUsage.totalTokenCount }}

      English Brevity Agent:
      - Input tokens: {{ outputs.english_brevity.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.english_brevity.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.english_brevity.tokenUsage.totalTokenCount }}

      💡 Tip: Monitor token usage to understand costs and optimize prompts!
pluginDefaults:
  - type: io.kestra.plugin.ai.agent.AIAgent
    values:
      provider:
        type: io.kestra.plugin.ai.provider.OpenAI
        modelName: gpt-5.4-mini
        apiKey: "{{ secret('OPENAI_API_KEY') }}"

INFO 2026-07-05T13:12:57.398330Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 279
- Output tokens: 93
- Total tokens: 372

English Brevity Agent:
- Input tokens: 108
- Output tokens: 63
- Total tokens: 171

💡 Tip: Monitor token usage to understand costs and optimize prompts!


Question 4

id: 4_simple_agent
namespace: zoomcamp
description: |
  This flow demonstrates a basic AI agent that summarizes text with controllable length and language. It shows:
  - How to structure agent prompts
  - How to chain multiple agent tasks
  - How to use pluginDefaults to avoid repetition
  - How to track token usage for cost monitoring
inputs:
  - id: summary_length
    displayName: Summary Length
    type: SELECT
    defaults: medium
    values:
      - short
      - medium
      - long
  - id: language
    displayName: Language
    type: SELECT
    defaults: en
    values:
      - en
      - fr
      - de
      - es
      - it
      - pt
      - ja
  - id: text
    type: STRING
    displayName: Text to summarize
    defaults: |
      Kestra is an open-source orchestration platform that allows you to define workflows declaratively in YAML. It enables both developers and non-developers to automate tasks through a no-code interface, while keeping everything versioned, governed, secure, and auditable. Kestra extends easily for custom use cases through plugins and custom scripts.

      Kestra follows a "start simple and grow as needed" philosophy. You can schedule a basic workflow in a few minutes, then later add Python scripts, Docker containers, or complex branching logic if the situation requires it. This makes Kestra ideal for data engineering, ETL pipelines, business process automation, and more.

      In LLM Zoomcamp, we learn how to build production-ready LLM applications using RAG, vector search, agents, and evaluation. In this bonus module, we're exploring how AI can accelerate workflow development through AI Copilot, RAG, and autonomous agents.
tasks:
  - id: multilingual_agent
    type: io.kestra.plugin.ai.agent.AIAgent
    description: Generate summary in requested language and length
    systemMessage: |
      You are a precise technical assistant.
      Produce a {{ inputs.summary_length }} summary in {{ inputs.language }}.
      Keep it factual, remove fluff, and avoid marketing language.
      If the input is empty or non-text, return a one-sentence explanation.

      Output format guidelines:
      - For 'short': 1-2 sentences
      - For 'medium': 2-5 sentences
      - For 'long': 1-3 paragraphs
    prompt: |
      Summarize the following content: {{ inputs.text }}
  - id: english_brevity
    type: io.kestra.plugin.ai.agent.AIAgent
    prompt: |
      Generate exactly 3 sentence English summary of the following:
      "{{ outputs.multilingual_agent.textOutput }}"
  - id: log_token_usage
    type: io.kestra.plugin.core.log.Log
    message: |
      📊 Token Usage Summary:

      Multilingual Agent:
      - Input tokens: {{ outputs.multilingual_agent.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.multilingual_agent.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.multilingual_agent.tokenUsage.totalTokenCount }}

      English Brevity Agent:
      - Input tokens: {{ outputs.english_brevity.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.english_brevity.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.english_brevity.tokenUsage.totalTokenCount }}

      💡 Tip: Monitor token usage to understand costs and optimize prompts!
pluginDefaults:
  - type: io.kestra.plugin.ai.agent.AIAgent
    values:
      provider:
        type: io.kestra.plugin.ai.provider.OpenAI
        modelName: gpt-5.4-mini
        apiKey: "{{ secret('OPENAI_API_KEY') }}"

INFO 2026-07-05T13:14:37.318188Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 279
- Output tokens: 184
- Total tokens: 463

English Brevity Agent:
- Input tokens: 199
- Output tokens: 68
- Total tokens: 267

💡 Tip: Monitor token usage to understand costs and optimize prompts!

Question 5

id: 4_simple_agent
namespace: zoomcamp
description: |
  This flow demonstrates a basic AI agent that summarizes text with controllable length and language. It shows:
  - How to structure agent prompts
  - How to chain multiple agent tasks
  - How to use pluginDefaults to avoid repetition
  - How to track token usage for cost monitoring
inputs:
  - id: summary_length
    displayName: Summary Length
    type: SELECT
    defaults: medium
    values:
      - short
      - medium
      - long
  - id: language
    displayName: Language
    type: SELECT
    defaults: en
    values:
      - en
      - fr
      - de
      - es
      - it
      - pt
      - ja
  - id: text
    type: STRING
    displayName: Text to summarize
    defaults: |
      Kestra is an open-source orchestration platform that allows you to define workflows declaratively in YAML. It enables both developers and non-developers to automate tasks through a no-code interface, while keeping everything versioned, governed, secure, and auditable. Kestra extends easily for custom use cases through plugins and custom scripts.

      Kestra follows a "start simple and grow as needed" philosophy. You can schedule a basic workflow in a few minutes, then later add Python scripts, Docker containers, or complex branching logic if the situation requires it. This makes Kestra ideal for data engineering, ETL pipelines, business process automation, and more.

      In LLM Zoomcamp, we learn how to build production-ready LLM applications using RAG, vector search, agents, and evaluation. In this bonus module, we're exploring how AI can accelerate workflow development through AI Copilot, RAG, and autonomous agents.
tasks:
  - id: multilingual_agent
    type: io.kestra.plugin.ai.agent.AIAgent
    description: Generate summary in requested language and length
    systemMessage: |
      You are a precise technical assistant.
      Produce a {{ inputs.summary_length }} summary in {{ inputs.language }}.
      Keep it factual, remove fluff, and avoid marketing language.
      If the input is empty or non-text, return a one-sentence explanation.

      Output format guidelines:
      - For 'short': 1-2 sentences
      - For 'medium': 2-5 sentences
      - For 'long': 1-3 paragraphs
    prompt: |
      Summarize the following content: {{ inputs.text }}
  - id: english_brevity
    type: io.kestra.plugin.ai.agent.AIAgent
    prompt: |
      Generate exactly 3 sentence English summary of the following:
      "{{ outputs.multilingual_agent.textOutput }}"
  - id: log_token_usage
    type: io.kestra.plugin.core.log.Log
    message: |
      📊 Token Usage Summary:

      Multilingual Agent:
      - Input tokens: {{ outputs.multilingual_agent.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.multilingual_agent.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.multilingual_agent.tokenUsage.totalTokenCount }}

      English Brevity Agent:
      - Input tokens: {{ outputs.english_brevity.tokenUsage.inputTokenCount }}
      - Output tokens: {{ outputs.english_brevity.tokenUsage.outputTokenCount }}
      - Total tokens: {{ outputs.english_brevity.tokenUsage.totalTokenCount }}

      💡 Tip: Monitor token usage to understand costs and optimize prompts!
pluginDefaults:
  - type: io.kestra.plugin.ai.agent.AIAgent
    values:
      provider:
        type: io.kestra.plugin.ai.provider.OpenAI
        modelName: gpt-5.4-mini
        apiKey: "{{ secret('OPENAI_API_KEY') }}"

INFO 2026-07-05T13:24:47.048943Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 279
- Output tokens: 151
- Total tokens: 430

English Brevity Agent:
- Input tokens: 166
- Output tokens: 116
- Total tokens: 282

💡 Tip: Monitor token usage to understand costs and optimize prompts!

